In [2]:
import os
from pathlib import Path
import pandas as pd
# ── Config ──────────────────────────────────────────────────────────────────
METADATA_FOLDER = "../data/Target_ALS"   # change to your folder
IGNORE_COLS     = {"subject_id"}

# ── Helpers ──────────────────────────────────────────────────────────────────
READERS = {
    ".csv":  lambda p: pd.read_csv(p),
    ".tsv":  lambda p: pd.read_csv(p, sep="\t"),
    ".txt":  lambda p: pd.read_csv(p, sep="\t"),
    ".xlsx": lambda p: pd.read_excel(p),
    ".xls":  lambda p: pd.read_excel(p),
}

def load_metadata_files(folder: str) -> dict[str, pd.DataFrame]:
    """Return {filename: DataFrame} for every supported file in folder."""
    dfs = {}
    for path in sorted(Path(folder).iterdir()):
        reader = READERS.get(path.suffix.lower())
        if reader:
            try:
                dfs[path.name] = reader(path)
                print(f"  loaded {path.name}  {dfs[path.name].shape}")
            except Exception as e:
                print(f"  ⚠ skipped {path.name}: {e}")
    return dfs

files = load_metadata_files(METADATA_FOLDER)
print(f"\nTotal files loaded: {len(files)}")

  loaded SYNAPSE_METADATA_MANIFEST.tsv  (7, 10)
  loaded nhs_clinical_data.xlsx  (230, 56)
  loaded nhs_linus_aural_analytics_speech_vitals_data.xlsx  (1008, 33)


/tmp/ipykernel_958134/2583648106.py:10: DtypeWarning: Columns (0,3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  ".csv":  lambda p: pd.read_csv(p),


  loaded nhs_olink_proteomics_data.csv  (1066240, 30)
  loaded nhs_sequencing_data_files.xlsx  (3562, 11)
  loaded nhs_simoa_nfl_proteomics_data.xlsx  (313, 10)
  loaded nhs_wgs_metadata.xlsx  (83, 27)
  loaded nhs_zephyrx_spirometry_data.xlsx  (3679, 54)

Total files loaded: 8


In [4]:
def unique_values_summary(dfs: dict[str, pd.DataFrame], ignore: set[str]) -> pd.DataFrame:
    """
    For every column across all loaded DataFrames, collect all unique values.
    Returns a DataFrame indexed by column name with a 'unique_values' list column.
    """
    agg: dict[str, set] = {}
    for fname, df in dfs.items():
        for col in df.columns:
            if col in ignore:
                continue
            vals = df[col].dropna().astype(str).unique().tolist()
            agg.setdefault(col, set()).update(vals)

    summary = pd.DataFrame(
        {"unique_values": {col: sorted(vals) for col, vals in agg.items()}}
    )
    summary.index.name = "column"
    return summary

summary_df = unique_values_summary(files, IGNORE_COLS)
summary_df
summary_df.to_csv("target_als_metadata_summary.csv")